**Purpose:** FINAL LSTM optimizing Sharpe directly (technical only).

**Outputs:** `LSTMvia-sharpe_technical_weights.npy`, `final_cumret.png`, `{name}.csv`, `{prefix}_cumret.png`

**Notes:** paths resolve via `src.config` (run `pip install -e .` from the repo root first).


In [ ]:
from src.seeds import SEED_PORTFOLIO_LSTM_SHARPE_V3_2_2, SEED_PORTFOLIO_LSTM_SHARPE_V3_2_2_FINAL


In [ ]:
from src.config import PROJECT_ROOT


# lstm returns for black litterman inspiratuion

- In forecasting the prices of multiple sectoral ETFs, a single LSTM model is an optimal approach due to its ability to leverage shared economic factors influencing all sectors, while maintaining flexibility to adapt to sector-specific behaviors. Sector ETFs are often correlated through common macroeconomic drivers, such as interest rates or commodity prices, and using a unified model enables the sharing of learned representations across sectors, improving generalization and reducing overfitting, especially in cases where data for some sectors may be limited. By incorporating sector embeddings as an input feature, the LSTM model can capture unique sectoral dynamics while still benefiting from the broader, cross-sector knowledge. This approach aligns with multi-task learning principles, where a single model can simultaneously optimize for multiple tasks (sector predictions), balancing efficiency, scalability, and accuracy. Empirical evidence from financial forecasting supports the efficacy of such global models, which often outperform separate models due to their ability to generalize across correlated assets. Thus, a single LSTM model with sector embeddings is both a computationally efficient and statistically robust solution for multi-sector ETF price prediction.

- Hyperparameters were selected using an expanding-window walk-forward validation scheme. Three folds were constructed: training on 5, 6, and 7 years of historical data respectively, each followed by a 1-year validation period. The final model was trained on the full 8-year pre-test period and evaluated on a strictly out-of-sample 2-year test set (2023–2025). This preserves temporal order and avoids information leakage common in random cross-validation for time-series data (Hyndman & Athanasopoulos). Some sources: [
0df79,
[00001](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html?utm_source=chatgpt.com),
[00002](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=3104847),
4061c
]

- FALTA:

    - falar das features e transformacoes

    - cross validation ja falei mas confirmar se estar ok

    - tem early stopping? rever codigo

    - que metricas foram usadas para otimizacao e isso? n usar so rmse, IC é fixe pq da ideia de direcao e como os assets se comportam relativamente uns aos outros, mas rmse é importante para avaliar a qualidade da previsao em termos de valor absoluto

    - ideia é usar top N modelos para portfolio opimzation (media = preço, variancia = incerteza la para o black litterman) -> acabar por usar mais features, vantagem lstm vs drl, mas drl é mt mais complexo (O complexity bla bla) ent em termos de tempo é oq é é lidar....

    - missing featudares, como lida?

    - what is the loss

    - ns se gosto da loss nem do early stop definido

    - aos 20 trails ja bateu quase 1 sharpe

    - N=5? pq nao 3 ou 10

    - 83780? 4061c? 2b4c0?


---

# LSTM Portfolio Optimiser — Direct Sharpe Weights

End-to-end LSTM that **directly predicts portfolio weights** and is trained
with a differentiable negative-Sharpe loss. No Black–Litterman, no MSE targets.

**Pipeline**
1. Load data & build features
2. Define model, loss, and training loop
3. Hyperparameter search with Optuna (walk-forward CV, 3 folds)
4. Final test with mid-test refit (2023–2024, 2024–2025)
5. Export weights → PortfolioMetrics

**Walk-forward CV folds** (no data leakage)
| Fold | Train | Val |
|------|-------|-----|
| 1 | 2015-07 → 2020-06 | 2020-07 → 2021-06 |
| 2 | 2015-07 → 2021-06 | 2021-07 → 2022-06 |
| 3 | 2015-07 → 2022-06 | 2022-07 → 2023-06 |

Test is strictly out-of-sample: **2023-07 → 2025-06**.


## 1) Imports

In [ ]:
from __future__ import annotations

import copy
import json
import logging
import math
import os
import random
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import optuna
from optuna.samplers import TPESampler
from optuna.trial import TrialState

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)


## 2) Config

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────
DATASET_PATH = Path(str(PROJECT_ROOT / '03_portfolio/dataset.parquet'))
SPREADS_PATH = Path(str(PROJECT_ROOT / '01_data/aux/bid-ask_spread.json'))

# ── Universe ─────────────────────────────────────────────────────────────
ASSETS = ['XLB', 'XLC', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLRE', 'XLU', 'XLV', 'XLY']

# ── Feature buckets (Optuna picks one per bucket each trial) ─────────────
FEATURES = {
    'Price':      ['Close'],
    'OHLC':       ['Open', 'High', 'Low'],
    'Trend':      ['SMA5', 'SMA22', 'SMA63', 'SMA126', 'SMA252', 'MACD', 'ADX'],
    'Momentum':   ['RSI', 'MACD', 'CCI'],
    'Volume':     ['Volume', 'OBV', 'ADI', 'VolumeVariation'],
    'Volatility': ['BBUpper', 'BBLower', '5dayVolatility', '22dayVolatility',
                   '63dayVolatility', 'RatioVolatility'],
    'Market':     ['VIXIndexClose'],
}

# ── Optuna / study ───────────────────────────────────────────────────────
OUT_DIR      = 'v322'
N_TRIALS     = 0
MODEL_SEEDS  = [11, 22, 33]
FINAL_SEED   = SEED_PORTFOLIO_LSTM_SHARPE_V3_2_2_FINAL


## 3) Utilities

In [ ]:
def setup_logger(log_dir: str) -> logging.Logger:
    """File + console logger scoped to log_dir."""
    #os.makedirs(log_dir, exist_ok=True)
    logger = logging.getLogger(f'sharpe_lstm_{os.path.basename(log_dir)}')
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    fmt = logging.Formatter('[%(asctime)s] [%(levelname)s] %(message)s')
    ch = logging.StreamHandler()
    ch.setFormatter(fmt)
    logger.addHandler(ch)
    #fh = logging.FileHandler(os.path.join(log_dir, 'run.log'))
    #fh.setFormatter(fmt)
    #logger.addHandler(fh)
    logger.propagate = False
    return logger


def seed_everything(seed: int):
    """Global seed for reproducibility across Python, NumPy, and PyTorch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def portfolio_metrics_from_returns(rets: np.ndarray, freq: int = 252) -> Dict[str, float]:
    """Compute annualised Sharpe, return, vol, max drawdown from a daily return series."""
    rets = np.asarray(rets, dtype=np.float64)
    rets = rets[np.isfinite(rets)]
    if rets.size == 0:
        return dict(n_days=0, mean_daily=np.nan, ann_return=np.nan,
                    ann_vol=np.nan, sharpe=np.nan, cum_return=np.nan, max_drawdown=np.nan)
    wealth   = np.cumprod(1.0 + rets)
    mean_d   = float(np.mean(rets))
    vol_d    = float(np.std(rets, ddof=0))
    ann_ret  = float((1.0 + mean_d) ** freq - 1.0)
    ann_vol  = float(vol_d * freq ** 0.5)
    sharpe   = float(mean_d / vol_d * freq ** 0.5) if vol_d > 1e-12 else np.nan
    peak     = np.maximum.accumulate(wealth)
    max_dd   = float(np.min(wealth / peak - 1.0))
    return dict(n_days=int(rets.size), mean_daily=mean_d, ann_return=ann_ret,
                ann_vol=ann_vol, sharpe=sharpe, cum_return=float(wealth[-1] - 1.0),
                max_drawdown=max_dd)


def _save_cumulative_returns_plot(
    out_dir: str, prefix: str, dates: np.ndarray, port_rets: np.ndarray
):
    """Save a cumulative-wealth plot to out_dir/prefix_cumret.png."""
    os.makedirs(out_dir, exist_ok=True)
    valid = np.isfinite(port_rets)
    if valid.sum() == 0:
        return
    wealth = np.cumprod(1.0 + port_rets[valid])
    plt.figure(figsize=(10, 5))
    plt.plot(dates[valid], wealth)
    plt.title(f'{prefix} | portfolio cumulative return')
    plt.xlabel('date')
    plt.ylabel('wealth (1 = initial)')
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f'{prefix}_cumret.png'))
    plt.close()


## 4) Walk-forward splits

In [ ]:
# Expanding-window folds: each fold grows the training window by 1 year.
# The validation window is always the following fiscal year.
# Transforms are fit on the training window only (no lookahead leakage).
_CV_PERIODS = [
    ('2015-07-01', '2020-06-30', '2020-07-01', '2021-06-30'),
    ('2015-07-01', '2021-06-30', '2021-07-01', '2022-06-30'),
    ('2015-07-01', '2022-06-30', '2022-07-01', '2023-06-30'),
]


def make_walk_forward_splits(dates_index: pd.DatetimeIndex) -> List[Tuple[np.ndarray, np.ndarray]]:
    """Return list of (train_idx, val_idx) integer-position arrays."""
    splits = []
    for tr_start, tr_end, v_start, v_end in _CV_PERIODS:
        train_idx = np.where(
            (dates_index >= pd.Timestamp(tr_start)) & (dates_index <= pd.Timestamp(tr_end))
        )[0]
        val_idx = np.where(
            (dates_index >= pd.Timestamp(v_start)) & (dates_index <= pd.Timestamp(v_end))
        )[0]
        if len(train_idx) == 0 or len(val_idx) == 0:
            print(f'[SKIP SPLIT] train={tr_start}..{tr_end} val={v_start}..{v_end} '
                  f'| lens train={len(train_idx)} val={len(val_idx)}')
            continue
        splits.append((train_idx, val_idx))
    if not splits:
        raise ValueError('No valid walk-forward splits. Check dataset date coverage.')
    return splits


def split_train_for_early_stop(
    train_idx: np.ndarray,
    dates_index: pd.DatetimeIndex,
    es_years: int = 1,
    min_train_days: int = 252,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carve the last `es_years` off the training window as an early-stopping
    holdout. Returns (core_train_idx, early_stop_idx).
    """
    train_dates = dates_index[train_idx]
    es_start    = train_dates[-1] - pd.DateOffset(years=es_years) + pd.Timedelta(days=1)
    es_mask     = train_dates >= es_start
    core_idx    = train_idx[~es_mask]
    es_idx      = train_idx[es_mask]
    # fallback: hard split if either segment is too small
    if len(core_idx) < min_train_days or len(es_idx) < min_train_days // 2:
        cut      = min(max(min_train_days, len(train_idx) - 252 * es_years), len(train_idx) - 1)
        core_idx = train_idx[:cut]
        es_idx   = train_idx[cut:]
    if len(core_idx) == 0 or len(es_idx) == 0:
        raise ValueError('Failed to create non-empty early-stopping split.')
    return core_idx, es_idx


## 5) Feature engineering

In [ ]:
def pick_features_from_buckets(trial: optuna.Trial, features: Dict[str, List[str]]) -> Dict[str, Any]:
    """
    Optuna picks one feature per bucket per trial.
    OHLC and Market are binary use/don't-use choices.
    Volatility can pick a single indicator or Bollinger Bands (upper + lower).
    """
    picks: Dict[str, Any] = {}
    picks['Price'] = 'log_ret_1d'  # always included

    picks['OHLC'] = ['Open', 'High', 'Low'] if trial.suggest_categorical('use_ohlc', [0, 1]) else []

    for bucket in ['Trend', 'Momentum', 'Volume']:
        picks[bucket] = trial.suggest_categorical(f'pick_{bucket}', features[bucket])

    # Volatility: allow BBands as a paired choice
    vol_options  = [f for f in features['Volatility'] if f not in ('BBLower', 'BBUpper')]
    vol_options.append('BBands')
    vol_choice   = trial.suggest_categorical('pick_Volatility', vol_options)
    picks['Volatility'] = ['BBLower', 'BBUpper'] if vol_choice == 'BBands' else vol_choice

    picks['Market'] = 'VIXIndexClose' if trial.suggest_categorical('use_market', [0, 1]) else None
    return picks


def pick_features_from_best_params(
    best_params: Dict[str, Any], features: Dict[str, List[str]]
) -> Dict[str, Any]:
    """Reconstruct feature picks from a saved trial's params dict (no live trial needed)."""
    class _DummyTrial:
        def __init__(self, params): self.params = params
        def suggest_categorical(self, name, choices): return self.params[name]
    return pick_features_from_buckets(_DummyTrial(best_params), features)


def get_per_asset_feature_list(picks: Dict[str, Any]) -> List[str]:
    """Flatten picks dict into an ordered, deduplicated list of column names."""
    feats = ['log_ret_1d']
    for f in picks.get('OHLC', []):
        feats.append(f)
    for bucket in ['Trend', 'Momentum', 'Volume', 'Volatility']:
        val = picks[bucket]
        feats.extend(val if isinstance(val, list) else [val])
    seen, out = set(), []
    for f in feats:
        if f not in seen:
            out.append(f)
            seen.add(f)
    return out


def required_df_columns(
    assets: List[str], per_asset_feats: List[str], picks: Dict[str, Any]
) -> List[str]:
    """List all DataFrame columns consumed by build_raw_arrays."""
    cols = []
    for a in assets:
        cols.append(f'{a}_Close')
        for f in per_asset_feats:
            if f not in ('log_ret_1d',):
                cols.append(f'{a}_{f}')
    if picks.get('Market') is not None:
        cols.append(picks['Market'])
    return list(dict.fromkeys(cols))  # deduplicated, order-preserving


## 6) Raw arrays & transforms

In [ ]:
def build_raw_arrays(
    df: pd.DataFrame,
    assets: List[str],
    picks: Dict[str, Any],
    features: Dict[str, List[str]],
    logger: logging.Logger,
    spreads: Optional[np.ndarray] = None,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, List[str], Optional[np.ndarray]]:
    """
    Build raw numpy arrays from the dataset DataFrame.

    Returns
    -------
    X_raw   : [T, N, F]  raw per-asset features (before normalisation)
    y_raw   : [T, N]     next-day realized returns, net of half-spread cost
    y_mask  : [T, N]     1 where y_raw is valid and ETF was available
    feat_names: List[str]
    market_raw: [T, 1] or None
    """
    per_asset_feats = get_per_asset_feature_list(picks)
    cols = required_df_columns(assets, per_asset_feats, picks)

    if spreads is None:
        spreads = np.zeros(len(assets), dtype=np.float32)
    spreads = np.asarray(spreads, dtype=np.float32)
    if spreads.shape[0] != len(assets):
        raise ValueError(f'spreads length {len(spreads)} != n_assets {len(assets)}')

    missing = [c for c in cols if c not in df.columns]
    if missing:
        logger.error(f'Missing columns: {missing[:25]}')
        raise KeyError('Dataset missing required columns.')

    T, N, F = len(df), len(assets), len(per_asset_feats)
    closes  = np.zeros((T, N), dtype=np.float32)
    X_raw   = np.zeros((T, N, F), dtype=np.float32)

    for i, a in enumerate(assets):
        c = df[f'{a}_Close'].to_numpy(np.float32)
        closes[:, i] = c

        for j, f in enumerate(per_asset_feats):
            if f == 'log_ret_1d':
                lr = np.full_like(c, np.nan)
                valid = (c[1:] > 0) & (c[:-1] > 0) & np.isfinite(c[1:]) & np.isfinite(c[:-1])
                lr[1:][valid] = np.log(c[1:][valid] / c[:-1][valid])
                X_raw[:, i, j] = lr
            elif f in ('Open', 'High', 'Low'):
                raw = df[f'{a}_{f}'].to_numpy(np.float32)
                rel = np.full_like(raw, np.nan)
                valid = (raw > 0) & (c > 0) & np.isfinite(raw) & np.isfinite(c)
                rel[valid] = np.log(raw[valid] / c[valid])
                X_raw[:, i, j] = rel
            elif f.startswith('SMA'):
                # express as log(price / SMA) — mean-reversion signal
                sma = df[f'{a}_{f}'].to_numpy(np.float32)
                rel = np.full_like(sma, np.nan)
                valid = (sma > 0) & (c > 0) & np.isfinite(sma) & np.isfinite(c)
                rel[valid] = np.log(c[valid] / sma[valid])
                X_raw[:, i, j] = rel
            else:
                X_raw[:, i, j] = df[f'{a}_{f}'].to_numpy(np.float32)

    # y_raw: next-day net return (deduct half-spread as transaction cost)
    y_raw  = np.full((T, N), np.nan, dtype=np.float32)
    y_mask = np.zeros((T, N), dtype=bool)
    for i in range(N):
        c     = closes[:, i]
        valid = np.isfinite(c[:-1]) & np.isfinite(c[1:]) & (c[:-1] > 0) & (c[1:] > 0)
        raw_ret          = np.full(T - 1, np.nan, dtype=np.float32)
        raw_ret[valid]   = (c[1:][valid] / c[:-1][valid]) - 1.0
        net_ret          = np.where(
            np.isfinite(raw_ret),
            np.sign(raw_ret) * np.maximum(np.abs(raw_ret) - float(spreads[i]), 0.0),
            np.nan,
        ).astype(np.float32)
        y_raw[:-1, i]  = net_ret
        y_mask[:-1, i] = valid

    market = None
    if picks.get('Market') is not None:
        market = df[picks['Market']].to_numpy(np.float32).reshape(-1, 1)

    return X_raw, y_raw, y_mask, per_asset_feats, market


def fit_transforms_on_train(
    X_train: np.ndarray, feat_names: List[str], market_train: Optional[np.ndarray]
) -> Tuple[List[Dict], Optional[Dict]]:
    """
    Fit a per-feature z-score normaliser using the training window only.
    Returns a list of (mean, std) dicts per feature, plus optional market stats.
    """
    _, N, F = X_train.shape
    fitted = []
    for f_idx in range(F):
        vals = X_train[:, :, f_idx].ravel()
        vals = vals[np.isfinite(vals)]
        mu  = float(vals.mean()) if len(vals) > 0 else 0.0
        std = float(vals.std())  if len(vals) > 0 else 1.0
        fitted.append({'mu': mu, 'std': max(std, 1e-8)})
    market_ft = None
    if market_train is not None:
        vals = market_train.ravel()
        vals = vals[np.isfinite(vals)]
        mu   = float(vals.mean()) if len(vals) > 0 else 0.0
        std  = float(vals.std())  if len(vals) > 0 else 1.0
        market_ft = {'mu': mu, 'std': max(std, 1e-8)}
    return fitted, market_ft


def apply_transforms(
    X: np.ndarray,
    fitted: List[Dict],
    market: Optional[np.ndarray],
    market_ft: Optional[Dict],
) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    """Apply pre-fitted z-score transforms to the full dataset."""
    X_out = X.copy()
    for f_idx, ft in enumerate(fitted):
        X_out[:, :, f_idx] = (X[:, :, f_idx] - ft['mu']) / ft['std']
    X_out = np.nan_to_num(X_out, nan=0.0, posinf=0.0, neginf=0.0)
    m_out = None
    if market is not None and market_ft is not None:
        m_out = (market - market_ft['mu']) / market_ft['std']
        m_out = np.nan_to_num(m_out, nan=0.0, posinf=0.0, neginf=0.0)
    return X_out, m_out


def compute_sample_validity(
    X_raw: np.ndarray, y_mask: np.ndarray, lookback: int, min_valid_assets: int = 1
) -> np.ndarray:
    """
    For each time-step t, a sample is valid if:
    - all features in the lookback window are finite
    - at least min_valid_assets assets have a valid return at t
    Returns a boolean array of shape [T].
    """
    finite_feats = np.isfinite(X_raw).all(axis=2)  # [T, N]
    sample_ok    = np.zeros(len(X_raw), dtype=bool)
    for t in range(lookback - 1, len(X_raw) - 1):
        win_ok    = finite_feats[t - lookback + 1: t + 1].all()
        target_ok = int(y_mask[t].sum()) >= min_valid_assets
        sample_ok[t] = bool(win_ok and target_ok)
    return sample_ok


## 7) Dataset & DataLoader

In [ ]:
class SequenceDataset(Dataset):
    """
    Each sample:
        x     : [L, N, F]  feature window
        y     : [N]        next-day net returns
        avail : [N]        availability mask (1 = ETF was trading that day)
        t     : int        absolute time index (for alignment)
    """
    def __init__(
        self,
        X: np.ndarray,
        y: np.ndarray,
        y_mask: np.ndarray,
        valid_t: np.ndarray,
        lookback: int,
        market: Optional[np.ndarray] = None,
    ):
        self.X       = X
        self.y       = y
        self.y_mask  = y_mask
        self.valid_t = np.asarray(valid_t, dtype=np.int64)
        self.lookback = int(lookback)
        self.market  = market

    def __len__(self): return len(self.valid_t)

    def __getitem__(self, idx):
        t = int(self.valid_t[idx])
        x = self.X[t - self.lookback + 1: t + 1].copy()  # [L, N, F]

        if self.market is not None:
            m = self.market[t - self.lookback + 1: t + 1]
            m = np.repeat(m[:, None, :], x.shape[1], axis=1)  # broadcast across assets
            x = np.concatenate([x, m], axis=2)

        y    = self.y[t].copy()
        avail = self.y_mask[t].copy().astype(np.float32)

        x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
        y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)

        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32),
            torch.tensor(avail, dtype=torch.float32),
            torch.tensor(t, dtype=torch.long),
        )


def make_loader(
    X: np.ndarray, y: np.ndarray, y_mask: np.ndarray,
    valid_t: np.ndarray, lookback: int, batch_size: int,
    market: Optional[np.ndarray], shuffle: bool,
) -> DataLoader:
    ds = SequenceDataset(X=X, y=y, y_mask=y_mask, valid_t=valid_t,
                         lookback=lookback, market=market)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=False)


## 8) Model — WeightLSTM

In [ ]:
class CrossAssetMixer(nn.Module):
    """
    Optional Transformer encoder applied across the asset dimension after the
    per-asset LSTM. Lets the model share information between sectors.
    Set num_layers=0 to bypass (becomes nn.Identity).
    """
    def __init__(self, d_model: int, num_heads: int, dropout: float, num_layers: int):
        super().__init__()
        if num_layers <= 0:
            self.net = nn.Identity()
        else:
            enc_layer = nn.TransformerEncoderLayer(
                d_model=d_model, nhead=num_heads,
                dim_feedforward=max(4 * d_model, 64),
                dropout=dropout, batch_first=True,
                activation='gelu', norm_first=True,
            )
            self.net = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class WeightLSTM(nn.Module):
    """
    Shared-encoder LSTM that outputs portfolio weights directly.

    Architecture
    ------------
    1. Per-asset LSTM encoder (weights shared across assets)
    2. Optional linear projection per asset
    3. CrossAssetMixer (Transformer) over asset embeddings
    4. MLP head → logits → weights

    weight_mode
    -----------
    'softmax'       long-only, sum=1 over available assets
    'unconstrained' signed weights, L1-normalised (allows short)
    """
    def __init__(
        self,
        input_dim_per_asset: int,
        num_assets: int,
        hidden_size: int,
        num_layers: int,
        dropout: float,
        bidirectional: bool = False,
        encoder_proj: int = 0,
        asset_mixer_layers: int = 1,
        asset_mixer_heads: int = 4,
        fusion_hidden: int = 64,
        weight_mode: str = 'softmax',
    ):
        super().__init__()
        self.num_assets  = num_assets
        self.weight_mode = weight_mode

        lstm_dropout = dropout if num_layers > 1 else 0.0
        self.encoder = nn.LSTM(
            input_size=input_dim_per_asset, hidden_size=hidden_size,
            num_layers=num_layers, dropout=lstm_dropout,
            batch_first=True, bidirectional=bidirectional,
        )
        enc_dim = hidden_size * (2 if bidirectional else 1)

        if encoder_proj > 0:
            self.asset_proj = nn.Sequential(
                nn.Linear(enc_dim, encoder_proj), nn.GELU(), nn.Dropout(dropout)
            )
            asset_dim = encoder_proj
        else:
            self.asset_proj = nn.Identity()
            asset_dim = enc_dim

        # ensure num_heads divides asset_dim
        if asset_mixer_layers > 0:
            valid_heads = [h for h in [1, 2, 4, 8] if asset_dim % h == 0]
            if not valid_heads:         asset_mixer_heads = 1
            elif asset_mixer_heads not in valid_heads: asset_mixer_heads = max(valid_heads)

        self.asset_mixer = CrossAssetMixer(
            d_model=asset_dim,
            num_heads=asset_mixer_heads if asset_mixer_layers > 0 else 1,
            dropout=dropout, num_layers=asset_mixer_layers,
        )

        fusion_in = num_assets * asset_dim
        self.head = (
            nn.Sequential(
                nn.Linear(fusion_in, fusion_hidden), nn.GELU(),
                nn.Dropout(dropout), nn.Linear(fusion_hidden, num_assets),
            )
            if fusion_hidden > 0
            else nn.Linear(fusion_in, num_assets)
        )

    def forward(self, x: torch.Tensor, avail: torch.Tensor) -> torch.Tensor:
        """
        x:     [B, L, N, F]
        avail: [B, N]  — 1 if asset available, 0 otherwise
        returns w: [B, N]  — portfolio weights (0 for unavailable assets)
        """
        B, L, N, F = x.shape
        # flatten batch × assets for shared LSTM pass
        x   = x.permute(0, 2, 1, 3).contiguous().reshape(B * N, L, F)
        out, _ = self.encoder(x)
        h   = self.asset_proj(out[:, -1, :])  # take last hidden state
        h   = h.reshape(B, N, -1)
        z   = self.asset_mixer(h)              # [B, N, asset_dim]
        logits = self.head(z.reshape(B, -1))   # [B, N]
        return self._to_weights(logits, avail)

    def _to_weights(self, logits: torch.Tensor, avail: torch.Tensor) -> torch.Tensor:
        mask_val = torch.finfo(logits.dtype).min / 2
        masked   = logits + (1.0 - avail) * mask_val
        if self.weight_mode == 'softmax':
            w = torch.softmax(masked, dim=-1) * avail
        elif self.weight_mode == 'unconstrained':
            w = masked * avail
            w = w / w.abs().sum(dim=-1, keepdim=True).clamp(min=1e-8)
        else:
            raise ValueError(f'Unknown weight_mode: {self.weight_mode}')
        return w


## 9) Loss & training loop

In [ ]:
def sharpe_loss(
    weights: torch.Tensor,
    returns: torch.Tensor,
    avail: torch.Tensor,
    eps: float = 1e-8,
    annualize: bool = False,
    freq: int = 252,
) -> torch.Tensor:
    """
    Differentiable negative Sharpe ratio (minimise → maximise Sharpe).

    Swap this function to change the training objective (e.g. Sortino, Calmar).

    Parameters
    ----------
    weights : [B, N]  already masked & normalised portfolio weights
    returns : [B, N]  realized next-day returns
    avail   : [B, N]  availability mask
    """
    port_ret = (weights * returns * avail).sum(dim=-1)  # [B]
    mean_ret = port_ret.mean()
    std_ret  = port_ret.std(unbiased=False).clamp(min=eps)
    sharpe   = mean_ret / std_ret
    if annualize:
        sharpe = sharpe * (freq ** 0.5)
    if not torch.isfinite(sharpe):
        raise ValueError('Sharpe became non-finite.')
    return -sharpe


@torch.no_grad()
def evaluate_model(
    model: WeightLSTM, loader: DataLoader, device: torch.device,
) -> Dict[str, Any]:
    """Run inference on a DataLoader and return portfolio metrics + raw arrays."""
    model.eval()
    port_rets, weights_all, t_idx_all = [], [], []
    for xb, yb, avail_b, tb in loader:
        xb, yb, avail_b = xb.to(device), yb.to(device), avail_b.to(device)
        w  = model(xb, avail_b)
        pr = (w * yb * avail_b).sum(dim=-1)
        port_rets.append(pr.cpu().numpy())
        weights_all.append(w.cpu().numpy())
        t_idx_all.append(tb.numpy())
    port_rets   = np.concatenate(port_rets)    if port_rets   else np.array([])
    weights_all = np.concatenate(weights_all)  if weights_all else np.empty((0, 0))
    t_idx_all   = np.concatenate(t_idx_all)    if t_idx_all   else np.array([], dtype=np.int64)
    return {'metrics': portfolio_metrics_from_returns(port_rets),
            'port_rets': port_rets, 'weights': weights_all, 't_idx': t_idx_all}


def train_one_model(
    X: np.ndarray, y: np.ndarray, y_mask: np.ndarray,
    market: Optional[np.ndarray],
    train_t: np.ndarray, es_t: np.ndarray,
    lookback: int, input_dim_per_asset: int, num_assets: int,
    params: Dict[str, Any], seed: int, logger: logging.Logger,
) -> Tuple[WeightLSTM, Dict]:
    """
    Train WeightLSTM with early stopping on the early-stop holdout window.
    Returns the best model (lowest es_neg_sharpe) and the training history.
    """
    seed_everything(seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    train_loader = make_loader(X, y, y_mask, train_t, lookback, params['batch_size'], market, shuffle=True)
    es_loader    = make_loader(X, y, y_mask, es_t,    lookback, params['batch_size'], market, shuffle=False)

    model = WeightLSTM(
        input_dim_per_asset=input_dim_per_asset, num_assets=num_assets,
        hidden_size=params['hidden_size'],  num_layers=params['num_layers'],
        dropout=params['dropout'],          bidirectional=params['bidirectional'],
        encoder_proj=params['encoder_proj'],
        asset_mixer_layers=params['asset_mixer_layers'],
        asset_mixer_heads=params['asset_mixer_heads'],
        fusion_hidden=params['fusion_hidden'], weight_mode=params['weight_mode'],
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=params['learning_rate'], weight_decay=params['weight_decay']
    )

    # optional LR scheduler
    if params['scheduler'] == 'plateau':
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=3
        )
    else:
        scheduler = None  # 'none' or unsupported value → no scheduler

    best_state, best_score, best_epoch = None, np.inf, -1
    patience_count = 0
    hist = {'train_loss': [], 'es_sharpe': []}

    for epoch in range(1, params['max_epochs'] + 1):
        model.train()
        batch_losses = []
        for xb, yb, avail_b, _ in train_loader:
            xb, yb, avail_b = xb.to(device), yb.to(device), avail_b.to(device)
            if not torch.isfinite(xb).all():
                raise ValueError('Non-finite input features.')
            if avail_b.sum() <= 0:
                continue
            optimizer.zero_grad()
            w    = model(xb, avail_b)
            if not torch.isfinite(w).all():
                raise ValueError('Non-finite weights from model.')
            loss = sharpe_loss(w, yb, avail_b, eps=params['sharpe_eps'])
            if not torch.isfinite(loss):
                raise ValueError('Sharpe loss became non-finite.')
            loss.backward()
            if params['grad_clip'] > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), params['grad_clip'])
            optimizer.step()
            batch_losses.append(float(loss.item()))

        train_loss   = float(np.mean(batch_losses)) if batch_losses else np.nan
        es_eval      = evaluate_model(model, es_loader, device)
        es_sharpe    = es_eval['metrics']['sharpe']
        es_neg_sharpe = -es_sharpe if np.isfinite(es_sharpe) else np.inf

        hist['train_loss'].append(train_loss)
        hist['es_sharpe'].append(es_sharpe)
        logger.info(f'[TRAIN] seed={seed} epoch={epoch:03d} '
                    f'train_loss={train_loss:.6f} es_sharpe={es_sharpe:.4f}')

        if scheduler is not None:
            scheduler.step(es_neg_sharpe)

        if np.isfinite(es_neg_sharpe) and es_neg_sharpe < (best_score - params['min_delta']):
            best_score = es_neg_sharpe
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            patience_count = 0
        else:
            patience_count += 1

        if epoch >= params['min_epochs'] and patience_count >= params['patience']:
            logger.info(f'[EARLY STOP] seed={seed} best_epoch={best_epoch} '
                        f'best_es_sharpe={-best_score:.4f}')
            break

    if best_state is None:
        raise ValueError('No valid best_state found during training.')
    model.load_state_dict(best_state)
    return model, hist


def train_and_eval_one_split_seed(
    X_transformed: np.ndarray, y: np.ndarray, y_mask: np.ndarray,
    market_transformed: Optional[np.ndarray],
    asset_names: List[str], dates_index: pd.DatetimeIndex,
    train_idx: np.ndarray, val_idx: np.ndarray,
    lookback: int, seed: int, model_params: Dict[str, Any],
    trial_dir: str, logger: logging.Logger, split_i: int,
    save_artifacts: bool = True,
) -> Dict[str, Any]:
    """Train + evaluate one (split, seed) combination. Called inside the Optuna objective."""
    train_core_idx, es_idx = split_train_for_early_stop(train_idx, dates_index)
    sample_valid = compute_sample_validity(
        X_transformed, y_mask, lookback,
        min_valid_assets=max(1, len(asset_names) // 3),
    )
    train_t = train_core_idx[sample_valid[train_core_idx]]
    es_t    = es_idx[sample_valid[es_idx]]
    val_t   = val_idx[sample_valid[val_idx]]

    if len(train_t) < 64 or len(es_t) < 32 or len(val_t) < 32:
        raise optuna.exceptions.TrialPruned()

    num_assets = len(asset_names)
    input_dim  = X_transformed.shape[2] + (1 if market_transformed is not None else 0)

    logger.info(f'[SPLIT {split_i}] seed={seed} '
                f'train={len(train_t)} es={len(es_t)} val={len(val_t)} '
                f'input_dim={input_dim} assets={num_assets}')

    model, _ = train_one_model(
        X=X_transformed, y=y, y_mask=y_mask, market=market_transformed,
        train_t=train_t, es_t=es_t, lookback=lookback,
        input_dim_per_asset=input_dim, num_assets=num_assets,
        params=model_params, seed=seed, logger=logger,
    )

    device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    val_loader = make_loader(
        X_transformed, y, y_mask, val_t, lookback,
        model_params['batch_size'], market_transformed, shuffle=False,
    )
    val_eval    = evaluate_model(model, val_loader, device)
    val_metrics = val_eval['metrics']

    logger.info(f'[VAL] split={split_i} seed={seed} '
                f"sharpe={val_metrics['sharpe']:.4f} "
                f"ann_ret={val_metrics['ann_return']:.4f} "
                f"ann_vol={val_metrics['ann_vol']:.4f} "
                f"max_dd={val_metrics['max_drawdown']:.4f}")

    if save_artifacts:
        art_dir = os.path.join(trial_dir, 'artifacts', f'split_{split_i}', f'seed_{seed}')
        os.makedirs(art_dir, exist_ok=True)
        _save_cumulative_returns_plot(
            out_dir=art_dir, prefix=f'split{split_i}_seed{seed}_val',
            dates=dates_index[val_eval['t_idx']], port_rets=val_eval['port_rets'],
        )

    return {'metrics': val_metrics, 'port_rets': val_eval['port_rets'],
            'weights': val_eval['weights'], 't_idx': val_eval['t_idx']}


## 10) Optuna objective & study runner

In [ ]:
def objective_factory(
    df: pd.DataFrame,
    assets: List[str],
    features: Dict[str, List[str]],
    base_log_dir: str,
    seeds: List[int],
    spreads: Optional[np.ndarray] = None,
):
    """
    Build the Optuna objective function.

    Each trial:
    1. Samples hyperparameters (features + architecture + training)
    2. Loops over all walk-forward CV folds
    3. Within each fold, trains/evaluates over multiple seeds
    4. Returns mean Sharpe across folds × seeds
    """
    dates_index = pd.DatetimeIndex(df.index)
    splits      = make_walk_forward_splits(dates_index)

    def objective(trial: optuna.Trial) -> float:
        trial_dir = os.path.join(base_log_dir, f'trial_{trial.number:05d}')
        #os.makedirs(trial_dir, exist_ok=True)
        logger = setup_logger(trial_dir)
        logger.info('=' * 80)
        logger.info(f'TRIAL {trial.number} START')

        # ── Feature selection ──────────────────────────────────────────────
        picks          = pick_features_from_buckets(trial, features)
        per_asset_feats = get_per_asset_feature_list(picks)
        trial.set_user_attr('feature_set',    picks)
        trial.set_user_attr('per_asset_feats', per_asset_feats)

        # ── Architecture hyperparameters ───────────────────────────────────
        lookback           = trial.suggest_categorical('lookback',           [5, 10, 22])
        hidden_size        = trial.suggest_categorical('hidden_size',        [32, 64, 128])
        num_layers         = trial.suggest_int('num_layers',                 1, 2)
        dropout            = trial.suggest_float('dropout',                  0.0, 0.3)
        bidirectional      = bool(trial.suggest_categorical('bidirectional', [0, 1]))
        encoder_proj       = trial.suggest_categorical('encoder_proj',       [0, 32, 64])
        asset_mixer_layers = trial.suggest_categorical('asset_mixer_layers', [0, 1, 2])
        asset_mixer_heads  = trial.suggest_categorical('asset_mixer_heads',  [1, 2, 4])
        fusion_hidden      = trial.suggest_categorical('fusion_hidden',      [0, 64, 128])
        weight_mode        = trial.suggest_categorical('weight_mode',        ['softmax', 'unconstrained'])

        # ── Training hyperparameters ───────────────────────────────────────
        learning_rate = trial.suggest_float('learning_rate', 1e-4, 2e-3, log=True)
        weight_decay  = trial.suggest_float('weight_decay',  1e-6, 1e-3, log=True)
        batch_size    = trial.suggest_categorical('batch_size',  [32, 64, 128])
        max_epochs    = trial.suggest_categorical('max_epochs',  [20, 30, 40])
        patience      = trial.suggest_categorical('patience',    [5, 7])
        min_epochs    = trial.suggest_categorical('min_epochs',  [5, 8])
        grad_clip     = trial.suggest_categorical('grad_clip',   [0.5, 1.0, 2.0])
        scheduler     = trial.suggest_categorical('scheduler',   ['none', 'plateau'])
        sharpe_eps    = trial.suggest_categorical('sharpe_eps',  [1e-6, 1e-8])
        min_delta     = trial.suggest_categorical('min_delta',   [0.0, 1e-4])

        model_params = dict(
            hidden_size=hidden_size,          num_layers=num_layers,
            dropout=dropout,                  bidirectional=bidirectional,
            encoder_proj=encoder_proj,        asset_mixer_layers=asset_mixer_layers,
            asset_mixer_heads=asset_mixer_heads, fusion_hidden=fusion_hidden,
            weight_mode=weight_mode,          learning_rate=learning_rate,
            weight_decay=weight_decay,        batch_size=batch_size,
            max_epochs=max_epochs,            patience=patience,
            min_epochs=min_epochs,            grad_clip=grad_clip,
            scheduler=scheduler,              sharpe_eps=sharpe_eps,
            min_delta=min_delta,
        )
        logger.info(f'Feature picks: {picks}')
        logger.info(f'Model params:  {model_params}')

        X_raw, y_raw, y_mask, feat_names, market_raw = build_raw_arrays(
            df=df, assets=assets, picks=picks, features=features,
            logger=logger, spreads=spreads,
        )

        # ── Walk-forward CV loop ───────────────────────────────────────────
        all_sharpes = []
        for split_i, (train_idx, val_idx) in enumerate(splits):
            logger.info('-' * 80)
            logger.info(f'[SPLIT {split_i}] '
                        f'train {df.index[train_idx[0]]}..{df.index[train_idx[-1]]} | '
                        f'val   {df.index[val_idx[0]]}..{df.index[val_idx[-1]]}')

            # fit transforms on training window only (no lookahead)
            X_tr_raw        = X_raw[train_idx]
            mkt_tr_raw      = market_raw[train_idx] if market_raw is not None else None
            fitted, mkt_ft  = fit_transforms_on_train(X_tr_raw, feat_names, mkt_tr_raw)
            X_trans, mkt_trans = apply_transforms(X_raw, fitted, market_raw, mkt_ft)

            split_scores = []
            for seed in seeds:
                try:
                    res = train_and_eval_one_split_seed(
                        X_transformed=X_trans, y=y_raw, y_mask=y_mask,
                        market_transformed=mkt_trans,
                        asset_names=assets, dates_index=dates_index,
                        train_idx=train_idx, val_idx=val_idx,
                        lookback=lookback, seed=seed,
                        model_params=model_params,
                        trial_dir=trial_dir, logger=logger, split_i=split_i,
                        save_artifacts=False,
                    )
                except ValueError as e:
                    logger.info(f'[PRUNE] Non-finite training: {e}')
                    raise optuna.TrialPruned()

                sharpe = res['metrics']['sharpe']
                score  = float(sharpe) if np.isfinite(sharpe) else -999.0
                all_sharpes.append(score)
                split_scores.append(score)
                logger.info(f'[SPLIT {split_i}][SEED {seed}] '
                            f"sharpe={score:.4f} "
                            f"ann_ret={res['metrics']['ann_return']:.4f} "
                            f"max_dd={res['metrics']['max_drawdown']:.4f}")

            # intermediate report for Optuna pruner
            split_mean = float(np.mean(split_scores))
            trial.report(split_mean, step=split_i)
            if trial.should_prune():
                logger.info('[PRUNE] Optuna pruned this trial.')
                raise optuna.TrialPruned()

        final_score = float(np.mean(all_sharpes))
        trial.set_user_attr('all_sharpes',  [float(x) for x in all_sharpes])
        trial.set_user_attr('mean_sharpe',  final_score)
        logger.info('=' * 80)
        logger.info(f'TRIAL {trial.number} DONE | mean_sharpe={final_score:.4f}')
        return final_score

    return objective


def run_optuna_search(
    df: pd.DataFrame,
    assets: List[str],
    features: Dict[str, List[str]],
    out_dir: str,
    n_trials: int = 50,
    seeds: Optional[List[int]] = None,
    spreads: Optional[np.ndarray] = None,
):
    """Create (or resume) an Optuna study and run `n_trials` trials."""
    os.makedirs(out_dir, exist_ok=True)
    seeds = seeds or [11, 22, 33]

    storage_path = os.path.join(out_dir, 'sharpe_lstm.db')
    study = optuna.create_study(
        study_name='sharpe_lstm',
        direction='maximize',
        storage=f'sqlite:///{storage_path}',
        load_if_exists=True,
        sampler=TPESampler(seed=None, multivariate=True, group=True, constant_liar=True),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1),
    )

    objective = objective_factory(
        df=df, assets=assets, features=features,
        base_log_dir=out_dir, seeds=seeds, spreads=spreads,
    )
    study.optimize(objective, n_trials=n_trials, gc_after_trial=True, show_progress_bar=True)

    best = study.best_trial
    summary = {'best_value': best.value, 'best_params': best.params,
               'best_user_attrs': best.user_attrs, 'n_trials': len(study.trials)}
    with open(os.path.join(out_dir, 'best_trial.json'), 'w') as f:
        json.dump(summary, f, indent=2)

    print('\n=== BEST TRIAL ===')
    print('Mean val Sharpe:', best.value)
    print('Params:',          best.params)
    print('Feature set:',     best.user_attrs.get('feature_set'))
    print('DB:',              storage_path)
    return study


## 11) Search space inspector

In [ ]:
def estimate_search_space(study: optuna.Study) -> int:
    """
    Inspect the study's distributions and print a suggested N_TRIALS ceiling.

    Discrete combinations (categorical × int params) are counted exactly.
    Continuous params (float) are excluded from the count — they have infinite
    resolution, but a handful of trials already samples them reasonably.

    Suggested ceiling = ceil(sqrt(discrete_combos))
    The ×5 factor accounts for continuous params and seed variance.
    Nothing is applied automatically — use the output to set N_TRIALS.
    """
    if not study.trials:
        raise ValueError('Run at least 1 trial first so distributions are populated.')

    distributions     = study.trials[0].distributions
    discrete_combinations = 1
    continuous_params = []

    print('Search space:')
    for name, dist in distributions.items():
        if isinstance(dist, optuna.distributions.CategoricalDistribution):
            n = len(dist.choices)
            discrete_combinations *= n
            print(f'  {name:25s}  categorical   {n:>4d} choices')
        elif isinstance(dist, optuna.distributions.IntDistribution):
            n = (dist.high - dist.low) // dist.step + 1
            discrete_combinations *= n
            print(f'  {name:25s}  int           {n:>4d} levels  [{dist.low} .. {dist.high}]')
        elif isinstance(dist, optuna.distributions.FloatDistribution):
            continuous_params.append(name)
            scale = 'log' if dist.log else 'linear'
            print(f'  {name:25s}  float ({scale:6s})        [{dist.low:.2e} .. {dist.high:.2e}]  ← continuous, excluded')

    suggested = math.ceil(math.sqrt(discrete_combinations))
    print()
    print(f'Discrete combinations  : {discrete_combinations}')
    print(f'Continuous params      : {continuous_params}')
    print(f'Suggested N_TRIALS     : ceil(sqrt({discrete_combinations})) = {suggested}')
    print(f'  └─ treat this as a ceiling; more trials risk overfitting the val folds')
    return suggested


## 12) Final test evaluation (with mid-test refit)

In [ ]:
def make_anchor_idx_by_target_date_range(
    dates_index: pd.DatetimeIndex,
    target_start: Optional[str] = None,
    target_end: Optional[str] = None,
) -> np.ndarray:
    """Return integer indices for dates_index[1:] that fall within [target_start, target_end]."""
    anchor_idx  = np.arange(len(dates_index) - 1, dtype=np.int64)
    target_dates = dates_index[1:]
    mask = np.ones(len(anchor_idx), dtype=bool)
    if target_start: mask &= target_dates >= pd.Timestamp(target_start)
    if target_end:   mask &= target_dates <= pd.Timestamp(target_end)
    out = anchor_idx[mask]
    if len(out) == 0:
        raise ValueError(f'No anchors for target range {target_start}..{target_end}')
    return out


def get_top_n_completed_trials(study: optuna.Study, top_n: int = 5):
    """Return top-N completed trials sorted by descending value."""
    trials = [
        t for t in study.trials
        if t.state == TrialState.COMPLETE and t.value is not None and np.isfinite(t.value)
    ]
    trials.sort(key=lambda t: t.value, reverse=True)
    if not trials:
        raise ValueError('No completed Optuna trials found.')
    return trials[:top_n]


def refit_and_predict_block(
    df: pd.DataFrame,
    assets: List[str],
    features: Dict[str, List[str]],
    best_trial: optuna.trial.FrozenTrial,
    train_idx: np.ndarray,
    test_idx: np.ndarray,
    dates_index: pd.DatetimeIndex,
    seed: int,
    logger: logging.Logger,
    spreads: Optional[np.ndarray] = None,
) -> Dict[str, Any]:
    """
    Refit the best trial's architecture on `train_idx`, evaluate on `test_idx`.
    Returns weights and returns as DataFrames indexed by target date.
    """
    N      = len(assets)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    picks  = pick_features_from_best_params(best_trial.params, features)
    lookback = int(best_trial.params['lookback'])

    model_params = {k: best_trial.params[k] for k in [
        'hidden_size', 'num_layers', 'dropout', 'bidirectional', 'encoder_proj',
        'asset_mixer_layers', 'asset_mixer_heads', 'fusion_hidden', 'weight_mode',
        'learning_rate', 'weight_decay', 'batch_size', 'max_epochs', 'patience',
        'min_epochs', 'grad_clip', 'scheduler',
    ]}
    model_params['bidirectional'] = bool(model_params['bidirectional'])
    model_params['sharpe_eps']    = best_trial.params.get('sharpe_eps', 1e-8)
    model_params['min_delta']     = best_trial.params.get('min_delta',  0.0)

    X_raw, y_raw, y_mask, feat_names, market_raw = build_raw_arrays(
        df=df, assets=assets, picks=picks, features=features,
        logger=logger, spreads=spreads,
    )
    fitted, mkt_ft = fit_transforms_on_train(
        X_raw[train_idx],
        feat_names,
        market_raw[train_idx] if market_raw is not None else None,
    )
    X_trans, mkt_trans = apply_transforms(X_raw, fitted, market_raw, mkt_ft)

    sample_valid  = compute_sample_validity(X_trans, y_mask, lookback,
                                             min_valid_assets=max(1, N // 3))
    core_idx, es_idx   = split_train_for_early_stop(train_idx, dates_index)
    train_core_t       = core_idx[sample_valid[core_idx]]
    es_t               = es_idx[sample_valid[es_idx]]
    test_t             = test_idx[sample_valid[test_idx]]
    all_train_t        = train_idx[sample_valid[train_idx]]

    if len(train_core_t) < 64 or len(es_t) < 32 or len(test_t) < 32:
        raise ValueError(
            f'Not enough samples: train_core={len(train_core_t)} '
            f'es={len(es_t)} test={len(test_t)}'
        )

    input_dim = X_trans.shape[2] + (1 if mkt_trans is not None else 0)
    model, _  = train_one_model(
        X=X_trans, y=y_raw, y_mask=y_mask, market=mkt_trans,
        train_t=train_core_t, es_t=es_t, lookback=lookback,
        input_dim_per_asset=input_dim, num_assets=N,
        params=model_params, seed=seed, logger=logger,
    )

    def _run_eval(t_indices):
        loader = make_loader(X_trans, y_raw, y_mask, t_indices, lookback,
                             model_params['batch_size'], mkt_trans, shuffle=False)
        return evaluate_model(model, loader, device)

    def _to_df(eval_res, t_indices):
        """Convert eval output to (weights_df, returns_df) indexed by target date."""
        n_days  = len(t_indices)
        w_full  = np.full((n_days, N), np.nan, dtype=np.float32)
        pr_full = np.full(n_days,      np.nan, dtype=np.float32)
        t_to_pos = {int(t): pos for pos, t in enumerate(t_indices)}
        for i, t in enumerate(eval_res['t_idx']):
            pos = t_to_pos.get(int(t))
            if pos is not None:
                w_full[pos]  = eval_res['weights'][i]
                pr_full[pos] = eval_res['port_rets'][i]
        target_dates = dates_index[t_indices + 1]
        return (pd.DataFrame(w_full,  index=target_dates, columns=assets).sort_index(),
                pd.DataFrame({'portfolio_return': pr_full}, index=target_dates).sort_index())

    train_weights_df, train_returns_df = _to_df(_run_eval(all_train_t), all_train_t)
    test_weights_df,  test_returns_df  = _to_df(_run_eval(test_t),      test_t)

    test_pm  = portfolio_metrics_from_returns(test_returns_df['portfolio_return'].dropna().to_numpy())
    train_pm = portfolio_metrics_from_returns(train_returns_df['portfolio_return'].dropna().to_numpy())
    logger.info(f"[BLOCK] train sharpe={train_pm['sharpe']:.4f} | "
                f"test sharpe={test_pm['sharpe']:.4f} "
                f"ann_ret={test_pm['ann_return']:.4f} max_dd={test_pm['max_drawdown']:.4f}")

    return {'test_weights_df': test_weights_df, 'test_returns_df': test_returns_df,
            'test_metrics': test_pm,
            'train_weights_df': train_weights_df, 'train_returns_df': train_returns_df,
            'train_metrics': train_pm}


def final_test_eval_with_midtest_refit(
    df: pd.DataFrame,
    assets: List[str],
    features: Dict[str, List[str]],
    study: optuna.Study,
    out_dir: str,
    seed: int = SEED_PORTFOLIO_LSTM_SHARPE_V3_2_2,
    train_target_end:  str = '2023-06-30',
    test_year1_start:  str = '2023-07-01',
    test_year1_end:    str = '2024-06-30',
    test_year2_start:  str = '2024-07-01',
    test_year2_end:    str = '2025-06-30',
    spreads: Optional[np.ndarray] = None,
) -> Dict[str, Any]:
    """
    Final evaluation with a mid-test refit.

    Block 1: train on all data up to 2023-06-30, test 2023-07 → 2024-06
    Block 2: retrain expanding to 2024-06-30,   test 2024-07 → 2025-06
    Combined test: concatenated blocks (no overlap by construction).
    """
    final_dir   = os.path.join(out_dir, 'final_test')
    os.makedirs(final_dir, exist_ok=True)
    logger      = setup_logger(final_dir)
    dates_index = pd.DatetimeIndex(df.index)
    best_trial  = study.best_trial

    b1_train = make_anchor_idx_by_target_date_range(dates_index, target_end=train_target_end)
    b1_test  = make_anchor_idx_by_target_date_range(dates_index, target_start=test_year1_start,
                                                                  target_end=test_year1_end)
    b2_train = make_anchor_idx_by_target_date_range(dates_index, target_end=test_year1_end)
    b2_test  = make_anchor_idx_by_target_date_range(dates_index, target_start=test_year2_start,
                                                                  target_end=test_year2_end)
    logger.info(f'Block1 train..{train_target_end} | test {test_year1_start}..{test_year1_end}')
    logger.info(f'Block2 train..{test_year1_end}   | test {test_year2_start}..{test_year2_end}')

    block1 = refit_and_predict_block(df=df, assets=assets, features=features,
                                     best_trial=best_trial, train_idx=b1_train,
                                     test_idx=b1_test, dates_index=dates_index,
                                     seed=seed, logger=logger, spreads=spreads)
    block2 = refit_and_predict_block(df=df, assets=assets, features=features,
                                     best_trial=best_trial, train_idx=b2_train,
                                     test_idx=b2_test, dates_index=dates_index,
                                     seed=seed, logger=logger, spreads=spreads)

    test_weights_df = pd.concat([block1['test_weights_df'], block2['test_weights_df']]).sort_index()
    test_returns_df = pd.concat([block1['test_returns_df'], block2['test_returns_df']]).sort_index()
    train_weights_df = block1['train_weights_df']  # block1 train = reference
    train_returns_df = block1['train_returns_df']

    combined_pm = portfolio_metrics_from_returns(
        test_returns_df['portfolio_return'].dropna().to_numpy()
    )

    # save CSVs
    for name, obj in [('test_weights', test_weights_df), ('test_returns', test_returns_df),
                      ('train_weights', train_weights_df), ('train_returns', train_returns_df)]:
        obj.to_csv(os.path.join(final_dir, f'{name}.csv'))

    # cumulative return plot
    test_rets = test_returns_df['portfolio_return'].dropna().to_numpy()
    if len(test_rets) > 0:
        plt.figure(figsize=(12, 5))
        plt.plot(test_returns_df.dropna().index, np.cumprod(1.0 + test_rets))
        plt.title('Final test | portfolio cumulative return')
        plt.xlabel('date'); plt.ylabel('wealth')
        plt.tight_layout()
        plt.savefig(os.path.join(final_dir, 'final_cumret.png'))
        plt.close()

    result = {
        'config': {'seed': seed, 'train_target_end': train_target_end,
                   'test_year1': f'{test_year1_start}..{test_year1_end}',
                   'test_year2': f'{test_year2_start}..{test_year2_end}'},
        'block1_test_metrics':   block1['test_metrics'],
        'block2_test_metrics':   block2['test_metrics'],
        'combined_test_metrics': combined_pm,
        'block1_train_metrics':  block1['train_metrics'],
    }

    def _json_default(x):
        if isinstance(x, (np.floating, float)): return float(x)
        if isinstance(x, (np.integer, int)):    return int(x)
        return x

    with open(os.path.join(final_dir, 'final_result.json'), 'w') as f:
        json.dump(result, f, indent=2, default=_json_default)

    print('\n=== FINAL TEST ===')
    print(f"Block1 ({test_year1_start}..{test_year1_end}): {block1['test_metrics']}")
    print(f"Block2 ({test_year2_start}..{test_year2_end}): {block2['test_metrics']}")
    print(f'Combined:                                  {combined_pm}')
    print(f'\nFiles saved to: {final_dir}/')

    return {**result, 'test_weights_df': test_weights_df, 'test_returns_df': test_returns_df,
            'train_weights_df': train_weights_df, 'train_returns_df': train_returns_df}


## 13) Run

In [ ]:
# ── Load data ────────────────────────────────────────────────────────────
df = pd.read_parquet(str(DATASET_PATH))

with open(str(SPREADS_PATH), 'r') as f:
    spread_dict = json.load(f)
spreads = np.array([spread_dict[a] for a in ASSETS], dtype=np.float32)

# ── 1) Hyperparameter search ──────────────────────────────────────────────
study = run_optuna_search(
    df=df, assets=ASSETS, features=FEATURES,
    out_dir=OUT_DIR, n_trials=N_TRIALS,
    seeds=MODEL_SEEDS, spreads=spreads,
)

# ── Search space summary (informational — does not change N_TRIALS) ───────
suggested = estimate_search_space(study)
total_n_trials = len(study.trials)
print(f'\nYou used N_TRIALS={total_n_trials}. Suggested ceiling: {suggested}.')

# ── 2) Final test with mid-test refit ─────────────────────────────────────
final_result = final_test_eval_with_midtest_refit(
    df=df, assets=ASSETS, features=FEATURES,
    study=study, out_dir=OUT_DIR, seed=FINAL_SEED,
    train_target_end='2023-06-30',
    test_year1_start='2023-07-01', test_year1_end='2024-06-30',
    test_year2_start='2024-07-01', test_year2_end='2025-06-30',
    spreads=spreads,
)

print('\n=== FINAL TEST RESULTS ===')
print(f"Block1 (2023-07..2024-06): {final_result['block1_test_metrics']}")
print(f"Block2 (2024-07..2025-06): {final_result['block2_test_metrics']}")
print(f"Combined:                  {final_result['combined_test_metrics']}")


In [ ]:
print(f'LSTM: You used N_TRIALS={total_n_trials}. Suggested ceiling: {suggested} x continuous.')


## 14) Portfolio metrics & plots

In [ ]:
weights = final_result['test_weights_df'][ASSETS].values


In [ ]:
import sys
from src.metrics import PortfolioMetrics

pm = PortfolioMetrics()
print(pm.summary(weights))
pm.plot_weights(weights)
pm.plot_cumulative_returns(weights)


In [ ]:
np.save("LSTMvia-sharpe_technical_weights.npy", weights)